In [1]:
!pip uninstall -y torchao
!pip install -q --no-cache-dir \
    transformers==4.51.3 \
    datasets==3.6.0 \
    accelerate==1.7.0 \
    peft==0.15.2 \
    trl==0.17.0

import trl, transformers, peft

print("trl:", trl.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 124.6 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 360.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 401.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 254.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 391.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 357.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 322.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 312.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 wh

In [18]:
import re
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
SYSTEM = (
    "You are an expert machine learning tutor. "
    "Answer clearly and concisely in English. "
    "Produce only one assistant answer."
)

def clean(x):
    return re.sub(r"\s+", " ", str(x)).strip()

book = pd.read_csv(
    "/kaggle/input/datasets/mohdmaaz036/qna-ml/"
    "ml_tutor_question_answer_700.csv"
)[["question", "answer"]].dropna()


# Book data: five questions belong to each concept
book["group"] = [f"book_{i // 5}" for i in range(len(book))]

# Use strong ML terms, not vague words such as data/train/model
ML_PATTERN = re.compile(
    r"machine learning|"
    r"deep learning|"
    r"neural network|"
    r"classification|"
    r"regression|"
    r"clustering|"
    r"overfitting|"
    r"underfitting|"
    r"gradient descent|"
    r"decision tree|"
    r"random forest|"
    r"support vector|"
    r"feature engineering|"
    r"cross.validation|"
    r"confusion matrix|"
    r"k.means|"
    r"naive bayes|"
    r"reinforcement learning|"
    r"natural language processing|"
    r"precision.recall|"
    r"precision score|"
    r"recall score|"
    r"classification metric|"
    r"true positive|"
    r"false positive|"
    r"false negative|"
    r"f1.score",
    re.I,
)

alpaca = load_dataset("tatsu-lab/alpaca", split="train")

# Keep only examples containing given ML terms & avoid vague words
alpaca = alpaca.filter(lambda x: bool(ML_PATTERN.search(clean(x["instruction"]) + " " + clean(x["input"]))))

# Randomly selects up to 1,300 relevant examples
alpaca = alpaca.shuffle(seed=SEED).select(range(min(1300, len(alpaca))))

general = pd.DataFrame(alpaca)

# Combining Instructions & Inputs
general["question"] = general.apply(lambda x: clean(x["instruction"]) + (f"\n\nInput: {clean(x['input'])}"
                        if clean(x["input"]) else ""), axis=1)

general["answer"] = general["output"].map(clean)

# This creates a unique group name for every Alpaca example
general["group"] = [f"alpaca_{i}" for i in range(len(general))]

general = general[["question", "answer", "group"]]

# Combine and clean
df = pd.concat([book, general], ignore_index=True)
df["question"] = df["question"].map(clean)
df["answer"] = df["answer"].map(clean)

# Remove non-English scripts, very short answers and duplicates
foreign = r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\u0e00-\u0e7f]"

df = df[

    # Removes rows containing Chinese, Japanese, Korean or Thai text
    ~df["question"].str.contains(foreign, regex=True) &
    ~df["answer"].str.contains(foreign, regex=True) &

    # Removes answers that are too short or extremely long
    df["answer"].str.split().str.len().between(8, 220)
].copy()

df["normalized_question"] = (df["question"].str.lower()

                             # Replaces punctuation and other non-word characters with spaces
                             .str.replace(r"\W+", " ", regex=True)
                             
                             .str.strip())

df = df.drop_duplicates("normalized_question").reset_index(drop=True)

Filter:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [33]:
BAD_ALPACA_GROUPS = {
    "alpaca_467",
    "alpaca_564",
    "alpaca_695",
    "alpaca_725",
    "alpaca_965",
}

before = len(df)

df = df[~df["group"].isin(BAD_ALPACA_GROUPS)].reset_index(drop=True)

after = len(df)

print("Removed:", before - after)
print("Remaining rows:", after)

Removed: 5
Remaining rows: 1612


In [34]:
from transformers import AutoTokenizer
from sklearn.model_selection import GroupShuffleSplit
from datasets import Dataset, DatasetDict


# =========================================================
# 1. TOKENIZER
# =========================================================

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


if tokenizer.pad_token_id is None:
    tokenizer.pad_token = "<|endoftext|>"

tokenizer.padding_side = "right"

# Qwen chat ending token
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("EOS token:", tokenizer.eos_token)
print("EOS ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD ID:", tokenizer.pad_token_id)
print("<|im_end|> ID:", IM_END_ID)


# =========================================================
# 2. TRAIN / VALIDATION / TEST SPLIT
# =========================================================

def split_groups(data, test_size, seed):
    
    splitter = GroupShuffleSplit(n_splits=1,  test_size=test_size,  random_state=seed)

    left, right = next(splitter.split( data,  groups=data["group"] ))

    return data.iloc[left].copy(), data.iloc[right].copy()


# 80% training
# 20% temporary
train_df, temporary_df = split_groups( df,  test_size=0.20,  seed=SEED )

# Temporary 20% →
# 10% validation + 10% test
validation_df, test_df = split_groups( temporary_df,  test_size=0.50,  seed=SEED + 1 )


print(
    f"Train: {len(train_df)} | "
    f"Validation: {len(validation_df)} | "
    f"Test: {len(test_df)}"
)


# =========================================================
# 3. CONVERT DATA TO CHAT FORMAT
# =========================================================

def make_dataset(frame):

    dataset = Dataset.from_pandas(frame[["question", "answer"]], preserve_index=False)

    def format_row(x):

        return {
            "prompt": [
                {
                    "role": "system",
                    "content": SYSTEM
                },
                {
                    "role": "user",
                    "content": x["question"]
                }
            ],

            "completion": [
                {
                    "role": "assistant",
                    "content": x["answer"].strip()
                }
            ]
        }

    return dataset.map(format_row, remove_columns=["question", "answer"])


# =========================================================
# 4. FINAL DATASETS
# =========================================================

splits = DatasetDict({

    "train": make_dataset(train_df),

    "validation": make_dataset(validation_df),

    "test": make_dataset(test_df),

})


print(splits)

EOS token: <|im_end|>
EOS ID: 151645
PAD token: <|endoftext|>
PAD ID: 151643
<|im_end|> ID: 151645
Train: 1293 | Validation: 161 | Test: 158


Map:   0%|          | 0/1293 [00:00<?, ? examples/s]

Map:   0%|          | 0/161 [00:00<?, ? examples/s]

Map:   0%|          | 0/158 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1293
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 161
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 158
    })
})


In [35]:
import gc
import torch

# Remove old trained model/trainer from memory
if "trainer" in globals():
    del trainer

if "model" in globals():
    del model

if "train_result" in globals():
    del train_result

if "test_dataset" in globals():
    del test_dataset

gc.collect()
torch.cuda.empty_cache()

print("✅ Old model cleared from memory")

✅ Old model cleared from memory


In [36]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig


# Fresh base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

# Training ke time cache OFF
model.config.use_cache = False

# Token IDs
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id


# LoRA configuration
peft_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


print("✅ Fresh model loaded")
print("Model:", MODEL_ID)
print("EOS ID:", model.config.eos_token_id)
print("PAD ID:", model.config.pad_token_id)
print("✅ LoRA config ready")

✅ Fresh model loaded
Model: Qwen/Qwen2.5-1.5B-Instruct
EOS ID: 151645
PAD ID: 151643
✅ LoRA config ready


In [37]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback


training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-ml-tutor-final-clean",

    max_length=512,

    num_train_epochs=3,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,

    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    weight_decay=0.01,
    max_grad_norm=1.0,

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    dataloader_num_workers=0,

    report_to="none",

    seed=SEED,

    completion_only_loss=True,

    packing=False,
)


trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=splits["train"],
    eval_dataset=splits["validation"],

    processing_class=tokenizer,

    peft_config=peft_config,
)


trainer.add_callback(
    EarlyStoppingCallback(
        early_stopping_patience=1
    )
)


print("✅ Final clean trainer ready")

trainer.model.print_trainable_parameters()

Converting train dataset to ChatML:   0%|          | 0/1293 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/1293 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1293 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1293 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/161 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


✅ Final clean trainer ready
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [38]:
for i in range(3):
    row = splits["train"][i]

    messages = row["prompt"] + row["completion"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    print(f"\n========== EXAMPLE {i+1} ==========\n")
    print(text)


========== EXAMPLE 1 ==========

<|im_start|>system
You are an expert machine learning tutor. Answer clearly and concisely in English. Produce only one assistant answer.<|im_end|>
<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
Machine learning builds programs that learn useful behavior from examples instead of requiring every decision rule to be written by hand. It is useful when the rules are too complicated to specify directly but representative examples are available.<|im_end|>


========== EXAMPLE 2 ==========

<|im_start|>system
You are an expert machine learning tutor. Answer clearly and concisely in English. Produce only one assistant answer.<|im_end|>
<|im_start|>user
Why do we use machine learning?<|im_end|>
<|im_start|>assistant
It is useful when the rules are too complicated to specify directly but representative examples are available. Use machine learning when you can define a prediction task, collect suitable data, and measure whether the pred

In [39]:
print("🚀 Starting FINAL clean training...")

train_result = trainer.train()

print("✅ Final training completed!")

🚀 Starting FINAL clean training...


Epoch,Training Loss,Validation Loss
0,1.482800,1.613178
1,1.212300,1.630412


✅ Final training completed!


In [40]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Best checkpoint: /kaggle/working/qwen-ml-tutor-final-clean/checkpoint-161
Best validation loss: 1.6131783723831177


In [41]:
test_dataset = trainer._prepare_dataset(
    splits["test"],
    processing_class=tokenizer,
    args=training_args,
    packing=False,
    formatting_func=None,
    dataset_name="test",
)

print("Test examples:", len(test_dataset))

Converting test dataset to ChatML:   0%|          | 0/158 [00:00<?, ? examples/s]

Applying chat template to test dataset:   0%|          | 0/158 [00:00<?, ? examples/s]

Tokenizing test dataset:   0%|          | 0/158 [00:00<?, ? examples/s]

Truncating test dataset:   0%|          | 0/158 [00:00<?, ? examples/s]

Test examples: 158


In [42]:
print("🧪 Evaluating FINAL model on test set...")

test_results = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

print("\nTest Results:")
print(test_results)

🧪 Evaluating FINAL model on test set...


early stopping required metric_for_best_model, but did not find eval_loss so early stopping is disabled



Test Results:
{'test_loss': 1.6767821311950684, 'test_runtime': 9.3341, 'test_samples_per_second': 16.927, 'test_steps_per_second': 8.464}


In [43]:
model = trainer.model
model.eval()

model.config.use_cache = True

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print("✅ Model ready for final testing")

✅ Model ready for final testing


In [44]:
import torch

def chat(question):

    messages = [
        {
            "role": "system",
            "content": SYSTEM
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.get_input_embeddings().weight.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [45]:
questions = [
    "What is the difference between a sample and a feature?",

    "What is the difference between precision and recall?",

    "When should I use lasso regression instead of ridge regression?",

    "What is the difference between a validation set and a test set?",

    "Why should the test set not be used for hyperparameter tuning?"
]


for i, question in enumerate(questions, 1):

    print(f"\n========== QUESTION {i} ==========")
    print("Question:", question)

    print("\nAnswer:")
    print(chat(question))


========== QUESTION 1 ==========
Question: What is the difference between a sample and a feature?

Answer:
A sample contains data for one or more observations, while a feature describes a characteristic of that observation. Samples can be used to train models, while features describe the characteristics of each sample.

========== QUESTION 2 ==========
Question: What is the difference between precision and recall?

Answer:
Precision measures how many of a model's positive predictions were actually correct, while recall measures how many of the actual positives were correctly identified by the model. Precision focuses on reducing false positives, while recall focuses on reducing false negatives.

========== QUESTION 3 ==========
Question: When should I use lasso regression instead of ridge regression?

Answer:
Use lasso when a sparse solution is important, such as for feature selection or to reduce overfitting. Lasso can remove some features entirely while keeping others, whereas ridge

In [46]:
FINAL_DIR = "/kaggle/working/qwen-ml-tutor-final"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("✅ Final model saved")
print("Location:", FINAL_DIR)

✅ Final model saved
Location: /kaggle/working/qwen-ml-tutor-final


In [47]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

FINAL_DIR = "/kaggle/working/qwen-ml-tutor-final"

# Hugging Face token
token = UserSecretsClient().get_secret("HF_WRITE_TOKEN")

api = HfApi(token=token)

REPO_ID = "mohd-maaz/qwen-ml-tutor-final"

# Create repo
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    private=True,
    exist_ok=True
)

# Upload final adapter + tokenizer
api.upload_folder(
    repo_id=REPO_ID,
    repo_type="model",
    folder_path=FINAL_DIR
)

print("✅ Final model uploaded to Hugging Face")
print("Repo:", REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Final model uploaded to Hugging Face
Repo: mohd-maaz/qwen-ml-tutor-final


In [48]:
import gc
import torch

for name in ["trainer", "model", "train_result", "test_dataset"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("✅ Old model cleared")

✅ Old model cleared


In [49]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_ID = "mohd-maaz/qwen-ml-tutor-final"

# Required because your repo is private
token = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
login(token=token)

# Tokenizer from your final repo
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_ID,
    token=token
)

# Fresh original Qwen model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
    token=token
)

# Attach your trained LoRA
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
    token=token
)

model.eval()
model.config.use_cache = True

print("✅ Hugging Face model reloaded successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/859 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

✅ Hugging Face model reloaded successfully


In [51]:
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

print("✅ Generation config cleaned")

✅ Generation config cleaned


In [53]:
SYSTEM = (
    "You are an expert machine learning tutor. "
    "Answer clearly and concisely in English. "
    "Produce only one assistant answer."
)

def chat(question):

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.get_input_embeddings().weight.device)

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print(chat(
    "What is the difference between precision and recall?"
))

Precision measures how many of a model's positive predictions were actually correct, while recall measures how many of the actual positives were correctly identified by the model. Precision focuses on reducing false positives, while recall focuses on reducing false negatives.
